# 02 - Step 1: Existing solutions
Reproduce what already exists, on **our own spatially held-out test points**:

| ID | Existing solution |
|---|---|
| E1 | Google Dynamic World (Sentinel-2, deep learning) |
| E2 | ESA WorldCover 2021 (Sentinel-1 + 2) |
| E3 | Esri Land Cover (Sentinel-2) |
| E4 | Traditional Random Forest, optical only |
| E5 | Random Forest, SAR only |

In [ ]:
# --- Setup (run once per Colab session) ---
!git clone -q https://github.com/veeresh0804/PrithviBodh.git 2>/dev/null || (cd PrithviBodh && git pull -q)
%cd /content/PrithviBodh
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/PrithviBodh'      # Earth Engine exports land here
import os; os.makedirs(DRIVE, exist_ok=True)

import sys; sys.path.insert(0, '.')
from prithvibodh.config import load_config
cfg = load_config()
cfg['gee_project'] = 'YOUR_EE_PROJECT_ID'         # <-- your free noncommercial project
from prithvibodh import gee_data as G
G.init(cfg)

In [ ]:
import pandas as pd
from prithvibodh.labels import load_labels
from prithvibodh import baselines, spatial_cv
lab = load_labels(f'{DRIVE}/labels.csv', cfg)
year, region = cfg['train_year'], G.aoi(cfg)

feats = G.sample_image(G.build_stack(cfg, year), lab)
prods = G.sample_image(baselines.all_products(cfg, region, year), lab)
df = lab.merge(feats, on='point_id').merge(prods, on='point_id', how='left')
df[['dynamic_world', 'worldcover', 'esri']] = df[['dynamic_world', 'worldcover', 'esri']].fillna(255)

df = spatial_cv.add_blocks(df, cfg['cv']['block_size_km'])
df = spatial_cv.split_blocks(df, cfg['cv']['test_fraction'], cfg['cv']['val_fraction'], cfg['seed'])
df.to_csv(f'{DRIVE}/points_table.csv', index=False)
df.split.value_counts()

In [ ]:
from prithvibodh import experiment, evaluate
rows_e, preds_e = experiment.run_existing(df, cfg)
existing = evaluate.results_table(rows_e)
existing.to_csv(f'{DRIVE}/results_existing.csv', index=False)
existing

In [ ]:
# Why spatial validation matters: the same optical RF with a RANDOM point split
from prithvibodh.spatial_cv import random_split
rows_r, _ = experiment.run_existing(random_split(df, .2, .2, cfg['seed']), cfg)
[r for r in rows_r if 'optical' in r['model']][0]   # compare macro_F1 with E4 above

In [ ]:
import matplotlib.pyplot as plt
test = df[df.split == 'test']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, ['dynamic_world', 'worldcover', 'esri']):
    ok = test[name] < 6
    evaluate.plot_confusion(test.label[ok], test[name][ok].astype(int), cfg['classes'], name, ax)
plt.tight_layout(); plt.savefig(f'{DRIVE}/confusion_existing.png', dpi=150)